# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q Faker
    print('Running in Colab')

# Descriptive Statistics In-Depth

This notebook goes beyond basic summaries to explore robust statistics, distribution shapes, and grouped analyses using a realistic synthetic dataset.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from faker import Faker

# Setup Faker and Seed
fake = Faker()
Faker.seed(42)
np.random.seed(42)

# 1. Generate Synthetic Social Science Data
n = 500
data = {
    'ID': [fake.uuid4() for _ in range(n)],
    'Age': np.random.randint(18, 80, n),
    'Gender': np.random.choice(['Male', 'Female', 'Non-binary'], n, p=[0.48, 0.48, 0.04]),
    'Income': np.random.lognormal(mean=10.5, sigma=0.6, size=n), # Log-normal distribution (skewed)
    'Education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n, p=[0.3, 0.4, 0.2, 0.1]),
    'Life_Satisfaction': np.random.randint(1, 8, n) # Likert scale 1-7
}

df = pd.DataFrame(data)
df['Income'] = df['Income'].round(2)

print("Dataset Head:")
print(df.head())

## 2. Advanced Summary Statistics
Beyond Mean and Standard Deviation, we look at:
*   **Median & MAD (Median Absolute Deviation)**: Robust measures for skewed data.
*   **Skewness & Kurtosis**: Quantifying distribution shape.

In [ ]:
# Standard Statistics
desc = df['Income'].describe()
print("--- Standard Statistics for Income ---")
print(desc)

# Robust Statistics
median = df['Income'].median()
mad = stats.median_abs_deviation(df['Income'])
skew = df['Income'].skew()
kurt = df['Income'].kurt()

print("\n--- Robust & Shape Statistics ---")
print(f"Median: {median:.2f}")
print(f"MAD:    {mad:.2f} (Robust dispersion)")
print(f"Skew:   {skew:.2f} (Positive = Right-skewed)")
print(f"Kurt:   {kurt:.2f} (High = Heavy tails)")

# Normality Test
shapiro_stat, shapiro_p = stats.shapiro(df['Income'])
print(f"\nShapiro-Wilk Test: p={shapiro_p:.5f} (p < 0.05 implies non-normal)")

## 3. Visualization of Distributions
Visualizing the skewness of Income and the discrete nature of Life Satisfaction.

In [ ]:
plt.figure(figsize=(14, 5))

# Histogram + KDE for Income
plt.subplot(1, 2, 1)
sns.histplot(df['Income'], kde=True, color='teal')
plt.axvline(df['Income'].mean(), color='r', linestyle='--', label='Mean')
plt.axvline(df['Income'].median(), color='gold', linestyle='-', label='Median')
plt.title('Income Distribution (Right-Skewed)')
plt.legend()

# Violin Plot for Life Satisfaction by Gender
plt.subplot(1, 2, 2)
sns.violinplot(x='Gender', y='Life_Satisfaction', data=df, palette='pastel')
plt.title('Life Satisfaction by Gender')

plt.tight_layout()
plt.show()

## 4. Grouped Statistics (Pivot Tables)
Comparing Income and Satisfaction across Education levels.

In [ ]:
# Group by Education
grouped = df.groupby('Education').agg({
    'Income': ['mean', 'median', 'std'],
    'Life_Satisfaction': ['mean', 'count']
}).round(2)

# Reorder index logically
edu_order = ['High School', 'Bachelor', 'Master', 'PhD']
grouped = grouped.reindex(edu_order)

print(grouped)

## 5. Pairplot
Exploring relationships between numeric variables.

In [ ]:
sns.pairplot(df[['Age', 'Income', 'Life_Satisfaction', 'Gender']], hue='Gender', diag_kind='kde')
plt.show()